In [42]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [43]:
from openai import OpenAI
groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = os.getenv("GROQ_API_KEY")
)

In [44]:
from sqlitesearch import TextSearchIndex
index = TextSearchIndex(
    text_fields= ['question','section','answer'],
    keyword_fields=['course'],
    db_path="database/FAQ.db"
)
index.search("just discovered the course, can i join now?")

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'e0a95572a6',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How do I join Slack if the invite email didn’t arrive?',
  'answer': 'Go to DataTalks.Club, request a Slack invite, or use the manual request form (processed daily). After joining, browse channels and join **#course-ml-zoomcamp**.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you

In [45]:
def search(query):
    boost_dict = {'question':3.0, 'section':.5}
    filter_dict = {'course': 'llm-zoomcamp'}
    
    return index.search(
        query,
        num_results = 5,
        boost_dict=boost_dict,
        filter_dict= filter_dict
    )

In [46]:
search_tool = {
    'type' : 'function',
    "name" : 'search',
    "description" : "Search the FAQ database for entries matching the given query.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "query" : {
                "type" : "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required" : ["query"],
        "additionalproperties" : False
    }
}

In [47]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [48]:
import json

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [49]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(823, 25)